In [11]:
import pandas as pd
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
df = diabetes.frame


In [12]:
df.head(1)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.05068,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0


In [13]:
df.dtypes

age       float64
sex       float64
bmi       float64
bp        float64
s1        float64
s2        float64
s3        float64
s4        float64
s5        float64
s6        float64
target    float64
dtype: object

In [14]:
df['sex'].unique()


array([ 0.05068012, -0.04464164])

In [15]:
file_path = '../data/raw/diabetes.csv'
df.to_csv(file_path, index=False)

In [16]:
import numpy as np
num_to_addNAN = int(len(df) * 0.10)
np.random.seed(42) 
age_indices = np.random.choice(df.index, size=num_to_addNAN, replace=False)
df.loc[age_indices, 'age'] = np.nan

bmi_indices = np.random.choice(df.index, size=num_to_addNAN, replace=False)
df.loc[bmi_indices, 'bmi'] = np.nan

print(df.isna().sum().to_dict())

{'age': 44, 'sex': 0, 'bmi': 44, 'bp': 0, 's1': 0, 's2': 0, 's3': 0, 's4': 0, 's5': 0, 's6': 0, 'target': 0}


In [17]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Определяем типы колонок
num_cols = ['age', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
cat_cols = ['sex']

In [19]:
class SquareFeaturesTransformer(BaseEstimator, TransformerMixin):
    """Добавляет квадраты числовых признаков"""
    def __init__(self, columns=None):
        self.columns = columns
        
    def fit(self, X, y=None):
        if self.columns is None:
            self.columns_ = [col for col in X.columns if X[col].dtype in ['float64', 'int64']]
        else:
            self.columns_ = self.columns
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        for col in self.columns_:
            X_copy[f'{col}_squared'] = X_copy[col] ** 2
        return X_copy

In [ ]:
num_transformer = Pipeline(steps=[
    ('square_features', SquareFeaturesTransformer(columns=num_cols)),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),      # числовые признаки
    ('cat', cat_transformer, cat_cols)        # категориальные признаки
])

param_grid = {
    'model__n_estimators': [50, 100, 150],
    'model__max_depth': [None, 5, 10],
    'model__min_samples_split': [2, 5]
}

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

grid_search = GridSearchCV(final_pipeline, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Лучшие параметры поиска: {grid_search.best_params_}")
print(f"RMSE на тестовой выборке: {rmse:.4f}")
print(f"R2 (коэффициент детерминации): {r2:.4f}")

Лучшие параметры поиска: {'model__max_depth': 5, 'model__min_samples_split': 5, 'model__n_estimators': 50}
RMSE на тестовой выборке: 53.9824
R2 (коэффициент детерминации): 0.4500


In [21]:
import joblib
import os

model_path = '../models/best_diabetes_model.pkl'
joblib.dump(best_model, model_path)

print(f"Модель сохранена в {model_path}")

Модель сохранена в ../models/best_diabetes_model.pkl
